### Read in phenotype manifest + data cleaning

In [2]:
import pandas as pd
import json

In [3]:
phenotype_manifest = pd.read_csv("phenotype_manifest.csv", usecols=[
    'description',
    'in_max_independent_set',
    'phenocode',
    'filename'
    ])

In [4]:
phenotype_manifest['description'] = phenotype_manifest['description'].fillna(phenotype_manifest['phenocode'])

### Read in excluded phenotypes

In [5]:
exclude_phenotypes = []

with open("exclude_phenotypes.txt", "r") as file:
    for line in file:
        if line[0]=='#':
            continue
        exclude_phenotypes.append(int(line.strip().split()[0]))

In [6]:
exclude_phenotypes

[113, 3553]

### Create filtering (maximally independent traits)

In [4]:
max_ind_set = phenotype_manifest[phenotype_manifest['in_max_independent_set']]

### Create filtering (ICD10 diseases)

In [7]:
with open("icd10_indices_n100.json", "r") as f:
    icd_indices = json.load(f)

In [8]:
icd_phenotypes = phenotype_manifest.iloc[icd_indices]

In [10]:
icd_phenotypes.head()

,phenocode,description,in_max_independent_set,filename
5466,41.1,Staphylococcus infections,False,phecode-041.1-both_sexes.tsv.bgz
5467,41.2,Streptococcus infection,False,phecode-041.2-both_sexes.tsv.bgz
5470,53.1,Herpes zoster with nervous system complications,False,phecode-053.1-both_sexes.tsv.bgz
5475,70.4,Chronic hepatitis,False,phecode-070.4-both_sexes.tsv.bgz
5476,70.9,Hepatitis NOS,False,phecode-070.9-both_sexes.tsv.bgz


In [31]:
print_list = [55, 119, 212, 213, 214, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 228, 229, 230] #, 121, 122, 123, 124, 125, 126, 127, 129, 131, 134, 136, 137, 138
# problematic_indices = []
# k = 1
# for index, row in icd_phenotypes.iterrows():
#     if k in print_list:
#         problematic_indices.append(index)
#         k+=1
#         continue
#     k+=1
print_list= [num-1 for num in print_list]
icd_phenotypes.iloc[print_list]

,phenocode,description,in_max_independent_set,filename
5590,204.22,"Myeloid leukemia, chronic",False,phecode-204.22-both_sexes.tsv.bgz
5698,275.3,Disorders of magnesium metabolism,False,phecode-275.3-both_sexes.tsv.bgz
5840,338.2,Chronic pain,False,phecode-338.2-both_sexes.tsv.bgz
5843,340.1,Migrain with aura,False,phecode-340.1-both_sexes.tsv.bgz
5849,345.1,Epilepsy,False,phecode-345.1-both_sexes.tsv.bgz
5851,345.12,Partial epilepsy,False,phecode-345.12-both_sexes.tsv.bgz
5852,345.3,Convulsions,False,phecode-345.3-both_sexes.tsv.bgz
5855,348.2,Cerebral edema and compression of brain,False,phecode-348.2-both_sexes.tsv.bgz
5856,348.4,Cerebral cysts,False,phecode-348.4-both_sexes.tsv.bgz
5857,348.7,Coma,False,phecode-348.7-both_sexes.tsv.bgz


### [DEPRECATED] Create .csv files from the track lists (for the deprecated lists)

In [ ]:
file_path = "DEPRECATED_track_lists_o3.json"
with open(file_path, "r") as json_file:
    depr_loaded_data = json.load(json_file)

In [ ]:
depr_file_name_dict = {}

In [ ]:
for key, value in depr_loaded_data.items():
    # Convert list to DataFrame
    df_temp = pd.DataFrame(value)

    # Create file name:
    if (not isinstance(key, str)):
        raise KeyError(f"expected description string as identifier but got {type(key)}")
    else:
        temp_file_name = "_".join(key.split()[:4])
        temp_file_name = temp_file_name.replace("(", "").replace(")", "").replace("/", "").replace("|", "").replace("-", "").replace("[", "").replace("]", "")
        temp_file_name = f"DEPRECATED_tracklists/{temp_file_name}.csv"
        depr_file_name_dict[key] = temp_file_name

    # Save to CSV without a header or index
    df_temp.to_csv(temp_file_name, index=False, header=False)

### Create .csv files from the track lists

In [10]:
file_path = "track_lists_o3.json"
with open(file_path, "r") as json_file:
    loaded_data = json.load(json_file)

In [11]:

for key, value in loaded_data.items():
    if not value:
        print(f"Key '{key}' has an empty list.")


In [ ]:
k=0
for key, value in loaded_data.items():
    # For testing purposes: only generate 5 phenotype csv's
    if k>5:
        continue

    # Skip excluded phenotypes
    if int(key) in exclude_phenotypes:
        print(f"Skipping excluded phenotype {key}.")
        continue

    # only generate csv's for relevant indices:
    if int(key) not in icd_indices:
        continue

    # Convert list to DataFrame
    df_temp = pd.DataFrame(value)

    # Create file name:
    if (not isinstance(key, str)):
        raise KeyError(f"expected phenotype manifest index as string but got {type(key)}")
    else:
        csv_file_name = "tracklists/tracks_phen" + key + ".csv"

    # Save to CSV without a header or index
    df_temp.to_csv(csv_file_name, index=False, header=False)

    k+=1

### Write .txt file (for maximally independent set)

The sumstats files are saved like this:

`/cluster/customapps/biomed/boeva/lrabuzin/continuous-103220-both_sexes.tsv.bgz`

The 1KG files are saved like this:

`/cluster/work/boeva/lrabuzin/1000genomes_as_csv`

The exon region files are saved like this: #TODO

In [ ]:
with open("method_parameters_independent_set.txt", "w") as file:
    for index, row in max_ind_set.iterrows():
        if index in exclude_phenotypes:
            print(f"Skipping excluded phenotype {index} {row['description']}.")
            continue
        file.write(f"""/cluster/customapps/biomed/boeva/lrabuzin/ukbb_phens/downloads/{row['filename']} 
                   /cluster/work/boeva/lrabuzin/1000genomes_as_csv
                   /cluster/work/boeva/lrabuzin/genome_assembly/exon_regions_v2.csv 
                   /cluster/work/boeva/lrabuzin/{file_name_dict[row['description']]}\n""")

### Write .txt file (for ICD10 diseases)

The sumstats files are saved like this:

`/cluster/customapps/biomed/boeva/lrabuzin/continuous-103220-both_sexes.tsv.bgz`

The 1KG files are saved like this:

`/cluster/work/boeva/lrabuzin/1000genomes_as_csv`

The exon region files are saved like this: #TODO

In [17]:
with open("method_icd_phenotypes.txt", "w") as file:
    for index, row in icd_phenotypes.iterrows():
        if index in exclude_phenotypes:
            print(f"Skipping excluded phenotype {index} {row['description']}.")
            continue
        file.write(f"""/cluster/customapps/biomed/boeva/lrabuzin/ukbb_phens/downloads/{row['filename']} 
                   /cluster/work/boeva/lrabuzin/1000genomes_as_csv
                   /cluster/work/boeva/lrabuzin/genome_assembly/exon_regions_v2.csv 
                   /cluster/work/boeva/lrabuzin/{"tracklists/tracks_phen" + str(index) + ".csv"}\n""")